# 05 — Đánh giá holdout | DenseNet121 (3-fold ensemble)

## 1. Môi trường và mã nguồn

In [3]:
import sys, subprocess, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 20260917
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

REPO_DIR = Path("/kaggle/working/The-Impostor-AI")
REPO_URL = "https://github.com/chienvoxn/The-Impostor-AI.git"
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "master", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "master"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))

from kemaodanh_oai.models import build_densenet121
from kemaodanh_oai.transforms import build_valid_transform
from kemaodanh_oai.dataset import FaceClassificationDataset
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
print("Project source imported successfully")

Device: cuda
Already up to date.
Project source imported successfully


From https://github.com/chienvoxn/The-Impostor-AI
 * branch            master     -> FETCH_HEAD


## 2. Tìm dữ liệu, split và checkpoint

In [4]:
from zipfile import ZipFile

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / "holdout_evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_pairs = [p for p in INPUT_DIR.rglob("pairs.csv") if p.parent.name == "train"]
if len(train_pairs) != 1:
    raise FileNotFoundError(f"Cần đúng một train/pairs.csv trong Kaggle Input, tìm thấy {len(train_pairs)}: {train_pairs}")
TRAIN_DIR = train_pairs[0].parent

# Ưu tiên split trong Kaggle Input; nếu không có, dùng split cố định từ repo.
split_candidates = list(INPUT_DIR.rglob("development_split.csv"))
if len(split_candidates) > 1:
    raise ValueError(f"Có nhiều development_split.csv, cần chọn một: {split_candidates}")
SPLIT_CSV = split_candidates[0] if split_candidates else REPO_DIR / "configs" / "development_split.csv"
assert SPLIT_CSV.is_file(), f"Không tìm thấy split: {SPLIT_CSV}"

# Tìm checkpoint trong Input (dataset/output notebook khác).
checkpoint_paths = {}
for fold in range(3):
    matches = list(INPUT_DIR.rglob(f"densenet121_fold{fold}_best.pt"))
    if len(matches) > 1:
        raise ValueError(f"Có nhiều checkpoint fold {fold}, hãy chỉ giữ một nguồn: {matches}")
    if matches:
        checkpoint_paths[fold] = matches[0]

if len(checkpoint_paths) < 3:
    zip_matches = list(INPUT_DIR.rglob("densenet121_3fold_results.zip"))
    if len(zip_matches) != 1:
        raise FileNotFoundError(
            "Không thấy đủ 3 checkpoint. Hãy Add Input output Notebook 04 "
            "hoặc dataset chứa densenet121_3fold_results.zip. "
            f"Checkpoint tìm thấy: {checkpoint_paths}; ZIP: {zip_matches}"
        )
    extract_dir = OUT_DIR / "imported_checkpoints"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with ZipFile(zip_matches[0]) as archive:
        for fold in range(3):
            target_name = f"densenet121_fold{fold}_best.pt"
            members = [x for x in archive.namelist() if Path(x).name == target_name]
            if len(members) != 1:
                raise ValueError(f"ZIP phải chứa đúng một {target_name}: {members}")
            import shutil
            dst = extract_dir / target_name
            with archive.open(members[0]) as src, dst.open("wb") as out:
                shutil.copyfileobj(src, out)
            checkpoint_paths[fold] = dst

print("Train directory:", TRAIN_DIR)
print("Split:", SPLIT_CSV)
for fold, path in sorted(checkpoint_paths.items()):
    print(f"Fold {fold} checkpoint:", path, f"({path.stat().st_size / 1024**2:.1f} MB)")

Train directory: /kaggle/input/datasets/chienvoo/dataset/the_imposter/the_imposter/data/train
Split: /kaggle/input/datasets/chienvoo/dataset/development_split.csv
Fold 0 checkpoint: /kaggle/input/datasets/chienvoo/densenet121-checkpoints/checkpoints/densenet121_fold0_best.pt (27.1 MB)
Fold 1 checkpoint: /kaggle/input/datasets/chienvoo/densenet121-checkpoints/checkpoints/densenet121_fold1_best.pt (27.1 MB)
Fold 2 checkpoint: /kaggle/input/datasets/chienvoo/densenet121-checkpoints/checkpoints/densenet121_fold2_best.pt (27.1 MB)


## 3. Lấy chính xác 200 cặp holdout, kiểm tra không trùng development

In [5]:
pairs_df = pd.read_csv(TRAIN_DIR / "pairs.csv", dtype={"pair_id": str})
split_df = pd.read_csv(SPLIT_CSV, dtype={"pair_id": str})
assert pairs_df["pair_id"].is_unique and split_df["pair_id"].is_unique
assert {"pair_id", "image_0", "image_1", "fake_position"}.issubset(pairs_df.columns)
assert {"pair_id", "split"}.issubset(split_df.columns)

merged = pairs_df.merge(split_df, on="pair_id", how="left", validate="one_to_one", indicator=True)
assert (merged["_merge"] == "both").all(), "Có pair thiếu trong split CSV"
assert set(merged["split"]) == {"development", "holdout"}, merged["split"].value_counts()
holdout_df = merged.loc[merged["split"] == "holdout"].sort_values("pair_id").reset_index(drop=True)
development_ids = set(merged.loc[merged["split"] == "development", "pair_id"])
assert len(merged) == 1000 and len(development_ids) == 800 and len(holdout_df) == 200
assert set(holdout_df["pair_id"]).isdisjoint(development_ids)
assert holdout_df["fake_position"].astype(int).isin([0, 1]).all()

records = []
for row in holdout_df.itertuples(index=False):
    for position in (0, 1):
        image_path = TRAIN_DIR / getattr(row, f"image_{position}")
        records.append({
            "pair_id": row.pair_id,
            "position": position,
            "image_path": str(image_path),
            "label": int(int(row.fake_position) == position),
        })
image_df = pd.DataFrame(records)
assert len(image_df) == 400
missing = [p for p in image_df["image_path"] if not Path(p).is_file()]
assert not missing, f"Thiếu ảnh, ví dụ: {missing[:3]}"
print("Development:", len(development_ids), "pairs")
print("Holdout:", len(holdout_df), "pairs /", len(image_df), "images")
print("Holdout label counts:", holdout_df["fake_position"].value_counts().to_dict())
print("No overlap with development: PASS")

Development: 800 pairs
Holdout: 200 pairs / 400 images
Holdout label counts: {0: 100, 1: 100}
No overlap with development: PASS


## 4. Inference với ba checkpoint (không huấn luyện)

In [6]:
dataset = FaceClassificationDataset(image_df, build_valid_transform(224, 0.60))
loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=2,
                    pin_memory=(DEVICE.type == "cuda"))

all_fold_probs = []
for fold in range(3):
    print(f"Running checkpoint fold {fold} ...", flush=True)
    model = build_densenet121(pretrained=False, num_classes=2).to(DEVICE)
    checkpoint = torch.load(checkpoint_paths[fold], map_location="cpu", weights_only=True)
    assert int(checkpoint["fold"]) == fold, f"Checkpoint không đúng fold {fold}"
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()

    probs_batches = []
    labels_batches = []
    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
                logits = model(images)
            probs_batches.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
            labels_batches.append(labels.numpy())
    probs = np.concatenate(probs_batches)
    labels = np.concatenate(labels_batches)
    assert len(probs) == len(image_df) and np.isfinite(probs).all()
    assert np.array_equal(labels, image_df["label"].to_numpy()), "Sai thứ tự ảnh holdout"
    all_fold_probs.append(probs)
    print(f"  Done. Best epoch: {checkpoint.get('epoch', 'N/A')}")
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

prob_matrix = np.stack(all_fold_probs, axis=0)  # [3, 400]
assert prob_matrix.shape == (3, 400)
print("Inference completed:", prob_matrix.shape)

Running checkpoint fold 0 ...
  Done. Best epoch: 7
Running checkpoint fold 1 ...
  Done. Best epoch: 10
Running checkpoint fold 2 ...
  Done. Best epoch: 20
Inference completed: (3, 400)


## 5. Ensemble và đánh giá holdout

In [7]:
prediction_df = image_df[["pair_id", "position"]].copy()
for fold in range(3):
    prediction_df[f"prob_fake_fold{fold}"] = prob_matrix[fold]
prediction_df["prob_fake_ensemble"] = prob_matrix.mean(axis=0)

wide = prediction_df.pivot(index="pair_id", columns="position", values="prob_fake_ensemble")
assert wide.shape == (200, 2) and wide.notna().all().all()
wide = wide.rename(columns={0: "prob_fake_0", 1: "prob_fake_1"})
truth = holdout_df.set_index("pair_id")["fake_position"].astype(int).rename("true_fake_position")
results = wide.join(truth, how="inner", validate="one_to_one")
assert len(results) == 200 and results.notna().all().all()
results["pred_fake_position"] = (results["prob_fake_1"] > results["prob_fake_0"]).astype(int)
results["correct"] = results["pred_fake_position"] == results["true_fake_position"]

true = results["true_fake_position"]
pred = results["pred_fake_position"]
macro_f1 = f1_score(true, pred, average="macro")
accuracy = accuracy_score(true, pred)
cm = confusion_matrix(true, pred, labels=[0, 1])

print("=" * 55)
print(f"HOLDOUT PAIR MACRO-F1: {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"HOLDOUT PAIR ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Correct pairs: {int(results['correct'].sum())}/200")
print("Confusion matrix (true rows, predicted columns):")
print(cm)
print(classification_report(true, pred, labels=[0, 1], zero_division=0))
display(results.head())

HOLDOUT PAIR MACRO-F1: 0.9800 (98.00%)
HOLDOUT PAIR ACCURACY: 0.9800 (98.00%)
Correct pairs: 196/200
Confusion matrix (true rows, predicted columns):
[[ 96   4]
 [  0 100]]
              precision    recall  f1-score   support

           0       1.00      0.96      0.98       100
           1       0.96      1.00      0.98       100

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200



,prob_fake_0,prob_fake_1,true_fake_position,pred_fake_position,correct
pair_id,,,,,
00003,0.989696,0.003522,0,0,True
00008,0.001971,0.986986,1,1,True
00023,0.077818,0.549172,1,1,True
00024,0.202474,0.996289,1,1,True
00029,0.068660,0.092653,0,1,False


## 6. Lưu kết quả và tải xuống

In [8]:
import json
import zipfile

prediction_path = OUT_DIR / "holdout_image_probabilities.csv"
results_path = OUT_DIR / "densenet121_holdout_predictions.csv"
metrics_path = OUT_DIR / "densenet121_holdout_metrics.json"
cm_path = OUT_DIR / "densenet121_holdout_confusion_matrix.csv"

prediction_df.to_csv(prediction_path, index=False)
results.reset_index().to_csv(results_path, index=False)
pd.DataFrame(cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"]).to_csv(cm_path)
metrics = {
    "dataset": "internal_holdout_from_train_pairs",
    "pairs": int(len(results)),
    "macro_f1": float(macro_f1),
    "accuracy": float(accuracy),
    "correct_pairs": int(results["correct"].sum()),
    "ensemble": "mean_class1_probability_across_three_densenet121_folds",
    "prediction_rule": "image_1_is_fake_if_prob_fake_1_greater_than_prob_fake_0",
    "confusion_matrix": cm.tolist(),
}
metrics_path.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = WORK_DIR / "densenet121_holdout_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for p in (prediction_path, results_path, metrics_path, cm_path):
        archive.write(p, arcname=p.name)
print("Saved:", zip_path)
print("Files:", [p.name for p in (prediction_path, results_path, metrics_path, cm_path)])
print("Download ZIP from Kaggle Output and save to local outputs/results/")

Saved: /kaggle/working/densenet121_holdout_results.zip
Files: ['holdout_image_probabilities.csv', 'densenet121_holdout_predictions.csv', 'densenet121_holdout_metrics.json', 'densenet121_holdout_confusion_matrix.csv']
Download ZIP from Kaggle Output and save to local outputs/results/
